### Step 1: Load the Silver Orders dataset

In [0]:
silver_df = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

display(silver_df.limit(10))

order_id,customer_id,customer_name,customer_email,customer_phone,product_id,product_name,quantity,unit_price,discount_percent,order_amount,final_amount,order_date,ship_date,delivery_date,shipping_state,pincode,ingestion_timestamp,source_file,batch_id,source_system,pipeline_id,payment_status,order_status,payment_method,category,shipping_city,invalid_customer_name,invalid_customer_email,invalid_customer_phone,invalid_order_id,invalid_customer_id,invalid_product_id,invalid_product_name,invalid_category,invalid_quantity,invalid_unit_price,invalid_discount_percent,invalid_order_amount,invalid_final_amount,invalid_payment_method,invalid_payment_status,invalid_order_status,invalid_order_date,invalid_ship_date,invalid_delivery_date,invalid_ship_date_order,invalid_delivery_date_ship,invalid_shipped_date,invalid_deliver_date,invalid_placed_dates,invalid_confirmed_delivery,invalid_returned_date,payment_order_status_violation,invalid_shipping_city,invalid_shipping_state,invalid_pincode_format,invalid_pincode,duplicate_order,reject_reason,record_valid
ORD100218,CUST10020,Zara Sundaram,zara.sundaram@yahoo.com,8249926919,PROD1064,Cricket Bat,8,5037.33,0,40298.64,40298.64,2025-10-18,2025-10-20,null,Karnataka,560073,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Shipped,NET BANKING,SPORTS,Bengaluru,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100164,CUST10042,Mahesh Shetty,mahesh.shetty@rediffmail.com,7678998028,PROD1073,Mouse,4,2091.76,80,8367.04,1673.41,2025-12-13,null,null,Uttar Pradesh,226094,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,NET BANKING,ELECTRONICS,Lucknow,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100064,CUST10136,Chitra Reddy,chitra.reddy@hotmail.com,8113585139,PROD1066,Books,4,2731.88,10,10927.52,9834.77,2025-11-27,null,null,Telangana,500032,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,DEBIT CARD,BOOKS,Hyderabad,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100596,CUST10067,Kabir Siddiqui,kabir.siddiqui@gmail.com,8812308209,PROD1036,Water Bottle,9,334.78,80,3013.02,602.6,2025-09-06,2025-09-07,2025-09-08,Maharashtra,411008,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,HOME,Pune,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100660,CUST10016,Radha Chandran,radha.chandran@rediffmail.com,9389854268,PROD1048,Bluetooth Speaker,5,7784.98,10,38924.9,35032.41,2025-01-25,2025-01-26,2025-01-30,Rajasthan,302000,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,ELECTRONICS,Jaipur,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100071,CUST10105,Rajesh Siddiqui,rajesh.siddiqui@rediffmail.com,6698509918,PROD1100,Mixer Grinder,6,8724.76,0,52348.56,52348.56,2025-11-28,2025-12-01,2025-12-02,West Bengal,700036,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,WALLET,HOME,Kolkata,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100463,CUST10087,Vikram Murthy,vikram.murthy@yahoo.com,8141078352,PROD1049,LED Monitor,9,38853.83,30,349

### Step 2: Check the available categories

In [0]:
silver_df.select("category").distinct().show(truncate=False)

+-----------+
|category   |
+-----------+
|GROCERY    |
|HOME       |
|BEAUTY     |
|SPORTS     |
|BOOKS      |
|FASHION    |
|ELECTRONICS|
+-----------+



### Step 3: Create a small lookup dataset

In [0]:
category_date = [
    ("GROCERY","Daily Essentials"),
    ("HOME","Home and Living"),
    ("BEAUTY","Beauty and Personal Care"),
    ("SPORTS","Sports and Fitness"),
    ("BOOKS","Books and Stationery"),
    ("FASHION","Clothing and Fashion"),
    ("ELECTRONICS","Electronics and Technology")
]

category_lookup_df = spark.createDataFrame(
    category_date,
    ['category','category_type']
)

display(category_lookup_df)

category,category_type
GROCERY,Daily Essentials
HOME,Home and Living
BEAUTY,Beauty and Personal Care
SPORTS,Sports and Fitness
BOOKS,Books and Stationery
FASHION,Clothing and Fashion
ELECTRONICS,Electronics and Technology


### Step 4: Perform a normal join

In [0]:
normal_join_df = silver_df.join(
    category_lookup_df,
    on="category",
    how="left"
)

display(
    normal_join_df
    .select(
        "order_id",
        "category",
        "category_type",
        "final_amount"
    ).limit(10)
)

order_id,category,category_type,final_amount
ORD100218,SPORTS,Sports and Fitness,40298.64
ORD100164,ELECTRONICS,Electronics and Technology,1673.41
ORD100064,BOOKS,Books and Stationery,9834.77
ORD100596,HOME,Home and Living,602.6
ORD100660,ELECTRONICS,Electronics and Technology,35032.41
ORD100071,HOME,Home and Living,52348.56
ORD100463,ELECTRONICS,Electronics and Technology,244779.13
ORD100606,BEAUTY,Beauty and Personal Care,5896.0
ORD100653,GROCERY,Daily Essentials,3475.28
ORD100266,FASHION,Clothing and Fashion,3191.9


### Step 5: Perform a Broadcast Join

In [0]:
from pyspark.sql.functions import broadcast

broadcast_join_df = silver_df.join(
    broadcast(category_lookup_df),
    on="category",
    how="left"
)

display(
    broadcast_join_df
    .select(
        "order_id",
        "category",
        "category_type",
        "final_amount"
    ).limit(10)
)

order_id,category,category_type,final_amount
ORD100218,SPORTS,Sports and Fitness,40298.64
ORD100164,ELECTRONICS,Electronics and Technology,1673.41
ORD100064,BOOKS,Books and Stationery,9834.77
ORD100596,HOME,Home and Living,602.6
ORD100660,ELECTRONICS,Electronics and Technology,35032.41
ORD100071,HOME,Home and Living,52348.56
ORD100463,ELECTRONICS,Electronics and Technology,244779.13
ORD100606,BEAUTY,Beauty and Personal Care,5896.0
ORD100653,GROCERY,Daily Essentials,3475.28
ORD100266,FASHION,Clothing and Fashion,3191.9


### Step 6: Inspect the execution plan

In [0]:
broadcast_join_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin LeftOuter (7)
            :- PhotonScan parquet workspace.ecommerce_project.silver_orders (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonRowToColumnar (3)
                        +- LocalTableScan (2)


(1) PhotonScan parquet workspace.ecommerce_project.silver_orders
Output [61]: [order_id#12072, customer_id#12073, customer_name#12074, customer_email#12075, customer_phone#12076, product_id#12077, product_name#12078, quantity#12079, unit_price#12080, discount_percent#12081, order_amount#12082, final_amount#12083, order_date#12084, ship_date#12085, delivery_date#12086, shipping_state#12087, pincode#12088, ingestion_timestamp#12089, source_file#12090, batch_id#12091, source_system#